<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# 🎧 AfriCare Support Analytics — DataProjectLab

## 📓 Notebook 3 — SQL Analytics, KPIs & Performance

---

> **🎯 Niveau :** Avancé | **⏱️ Durée :** 4-5h
> ❗ Ce notebook ne contient pas de solution.



In [ ]:
# pour executer du code SQL dans le notebook
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
# ================================
# 0. Imports & connexion
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import os, sys, warnings

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.2f}".format)

COLORS = {
    "primary": "#534AB7", "secondary": "#1D9E75",
    "warning": "#EF9F27", "danger":  "#E24B4A", "neutral": "#888780"
}

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/DataProjectLab/customer_support_analytics/"
else:
    SAVE_PATH = "./outputs/"
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"📁 Environnement : {'Colab' if IN_COLAB else 'Local'}")
print(f"📁 Dossier       : {SAVE_PATH}") 

# ── Chemins des fichiers ────────────────────────────────────────────────────
# Fichiers bruts depuis GitHub (ne changent pas)
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_support_analytics/data/"


# ── Connexion DuckDB ────────────────────────────────────────────────────────
# Les fichiers bruts (agents, categories) sont chargés depuis GitHub
# Le fichier nettoyé (tickets enrichis) est chargé depuis SAVE_PATH
con = duckdb.connect()
con.execute(f"""
    CREATE TABLE agents       AS SELECT * FROM read_csv_auto('{BASE_URL}tickets.csv');
    CREATE TABLE agents       AS SELECT * FROM read_csv_auto('{BASE_URL}agents.csv');
    CREATE TABLE categories   AS SELECT * FROM read_csv_auto('{BASE_URL}categories.csv');
    CREATE TABLE interactions AS SELECT * FROM read_csv_auto('{BASE_URL}interactions.csv');
    CREATE TABLE sla_alerts   AS SELECT * FROM read_csv_auto('{BASE_URL}sla_alerts.csv');
""")

print(f"✅ {con.execute('SELECT COUNT(*) FROM tickets').fetchone()[0]:,} tickets chargés")
print("Tables disponibles : tickets, agents, categories, interactions, sla_alerts")

In [ ]:
# Lier la connexion DuckDB à JupySQL pour les cellules %%sql
%load_ext sql
%sql con --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print("%%sql prêt ✅")

---

# 📊 1. KPIs globaux opérationnels

Calcule en **une seule requête** les indicateurs de pilotage principaux :

- `nb_tickets_total`, `nb_resolus`, `taux_resolution_pct`
- `taux_sla_breach_pct`, `taux_backlog_pct`, `taux_escalade_pct`
- `first_response_moy_h`, `resolution_moy_h`
- `csat_moyen` (attention aux NULLs !)
- `nb_rouverts`, `taux_reouverture_pct`

> 💡 Pour les taux : `ROUND(AVG(CASE WHEN sla_breach=1 THEN 1.0 ELSE 0.0 END)*100, 1)`

In [ ]:
%%sql df_kpi <<
    SELECT
        -- Votre code ici
    FROM tickets


print("KPIs GLOBAUX — AFRICAIRE SUPPORT")
print("="*55)
display(df_kpi)

### 🧠 Analyse

- Quel est le taux de SLA breach ? Est-il acceptable ?
- Quel est l'écart entre le premier contact et la résolution ?
- Le taux de réouverture est-il préoccupant ?

### 🧠 Tes observations

*(rédige tes conclusions ici)*

---

# 📊 2. Performance SLA par catégorie

Joint `tickets` avec `categories` et calcule par catégorie :
- `nb_tickets`, `taux_breach`, `resolution_moy_h`, `csat_moy`
- `ratio_sla_moy` : moyenne du ratio resolution/SLA
- `depassement_moy_h` : quand il y a breach, de combien d'heures en moyenne ?

Trie par `taux_breach` décroissant.

Crée **2 graphiques** :
1. Barres horizontales : taux de breach par catégorie (rouge si > 50%, orange si > 35%)
2. Scatter : SLA contractuel (X) vs Taux breach (Y), taille = nb tickets

> 💡 Le scatter révèle si les SLA courts (urgents) sont plus souvent dépassés que les SLA longs.

In [ ]:
%%sql df_sla_cat <<
    SELECT
        -- Votre code ici
    FROM tickets t
    JOIN categories c ON t.category_id = c.category_id
    GROUP BY c.nom, c.domaine, c.sla_heures, c.sla_strict
    ORDER BY taux_breach DESC

display(df_sla_cat)

In [ ]:
# Graphiques : barres + scatter
# Votre code ici

### 🧠 Tes observations

- Quelles catégories ont les taux de breach les plus élevés ?
- Y a-t-il une corrélation entre SLA court et taux de breach élevé ?

*(rédige ici)*

---

# 📊 3. Performance des agents — window functions ⭐

Joint `tickets` avec `agents` et calcule par agent :
- `nom`, `tier`, `bureau`
- `nb_tickets`, `taux_breach`, `resolution_moy_h`
- `csat_reel` : CSAT moyen observé vs `csat_moyen` historique (colonne agents)
- `ecart_csat` = csat_reel - csat_moyen (agent qui sur- ou sous-performe)
- `taux_reouverture`, `taux_escalade`
- `rank_sla` : classement par taux de breach (1 = meilleur) → **RANK() OVER**
- `rank_csat` : classement par CSAT réel → **RANK() OVER**

Crée **3 graphiques** :
1. Scatter : taux breach (X) vs CSAT moyen (Y) — taille = nb tickets — couleur = tier
2. Barres horizontales : nb tickets par agent (charge)
3. Heatmap agent × indicateur (taux breach / CSAT / taux réouverture)

> 💡 Un agent avec breach élevé ET CSAT faible est prioritaire pour le coaching.

In [ ]:
%%sql df_agt <<
    SELECT
        -- Votre code ici
        RANK() OVER (ORDER BY AVG(t.sla_breach) ASC)       AS rank_sla,
        RANK() OVER (ORDER BY AVG(t.csat) DESC NULLS LAST) AS rank_csat
    FROM tickets t
    JOIN agents a ON t.agent_id = a.agent_id
    GROUP BY a.nom, a.tier, a.bureau, a.csat_moyen
    ORDER BY taux_breach ASC


display(df_agt)

In [ ]:
# 3 graphiques : scatter + barres + heatmap
# Votre code ici

### 🧠 Tes observations

- Qui sont les top performers (faible breach + CSAT élevé) ?
- Qui nécessite un coaching immédiat ?
- Y a-t-il un lien entre le tier de l'agent et ses performances ?

*(rédige ici)*

---

# 📊 4. Analyse temporelle & heatmap

**Requête 4a — Évolution mensuelle :**
- `mois` · `nb_tickets` · `taux_sla_breach` · `resolution_moy_h` · `csat_moy`
- Variation du taux de breach via `LAG(taux_sla_breach) OVER (ORDER BY mois)`

**Requête 4b — Heatmap volume :**
- `jour_semaine` · `heure` (extraite de `created_at`) · `nb_tickets`

Crée **2 graphiques** :
1. Courbe mensuelle du taux breach avec bande de variance
2. Heatmap jour × heure (volume de tickets)

> 💡 Heure dans DuckDB : `HOUR(CAST(created_at AS TIMESTAMP))`

In [ ]:
%%sql df_evol <<
        
    WITH monthly AS (
        SELECT DATE_TRUNC('month', CAST(created_at AS TIMESTAMP)) AS mois,
               COUNT(*) AS nb,
               ROUND(AVG(CASE WHEN sla_breach=1 THEN 1.0 ELSE 0.0 END)*100, 1) AS breach_pct,
               ROUND(AVG(resolution_heures), 1) AS res_moy,
               ROUND(AVG(csat), 2) AS csat_moy
        FROM tickets
        GROUP BY 1
    )
    SELECT *,
           -- Ajouter LAG ici
    FROM monthly
    ORDER BY mois

display(df_evol)

In [ ]:
%%sql df_kpi <<
    SELECT
        jour_semaine,
        HOUR(CAST(created_at AS TIMESTAMP)) AS heure,
        COUNT(*) AS n
    FROM tickets
    GROUP BY jour_semaine, heure
    ORDER BY jour_semaine, heure

# Graphiques : courbe mensuelle + heatmap
# Votre code ici

### 🧠 Tes observations

- Y a-t-il une tendance haussière ou baissière sur le taux de breach ?
- Quelles heures et quels jours concentrent le plus de tickets ?
- Le weekend présente-t-il un risque particulier ?

*(rédige ici)*

---

# 📊 5. Analyse des alertes SLA et angles morts

**Requête 5a — Tickets en breach non notifiés :**
Joint `tickets` avec `sla_alerts`.
Identifie les tickets où `statut_notification = 'Non notifié'` ET `manager_alerte = False`.
Ce sont les **angles morts** du système d'alerte.

**Requête 5b — Distribution des dépassements par tranche :**
- Tranches : [0-2h] [2-8h] [8-24h] [1-3j] [3j+]
- Pour chaque tranche : nb tickets, csat_moy, taux_escalade

**Requête 5c — Nb moyen d'interactions avant escalade vs résolution normale :**
Joint `interactions` et `tickets`. Compare les deux populations.

Crée 2 graphiques : distribution des dépassements + comparaison interactions.

In [ ]:
# 5a — Angles morts (breach non notifiés)
%%sql df_angles <<
    SELECT
        -- Votre code ici
    FROM tickets

print(f"Tickets en breach non notifiés : {len(df_angles)}")
display(df_angles.head(10))

In [ ]:
# 5b — Distribution des dépassements par tranche
%%sql df_tranches <<
    SELECT
        -- Votre code ici
    FROM tickets


display(df_tranches)

In [ ]:
# 5c — Interactions avant escalade vs résolution normale
# Votre code ici

In [ ]:
# Graphiques : distribution dépassements + comparaison interactions
# Votre code ici

### 🧠 Tes observations

- Combien de tickets en breach sont passés sous le radar du système d'alerte ?
- Y a-t-il un lien entre le nombre d'interactions et l'escalade ?

*(rédige ici)*

---

# 📊 6. CSAT et satisfaction client

**Requête 6a — KPIs satisfaction :**
- Note moyenne globale
- Distribution des notes (count par rating 1-5)
- NPS simplifié : % Promoteurs (note 5) - % Détracteurs (note 1-2)

**Requête 6b — Impact du délai sur la satisfaction :**
- Groupe par tranche de résolution : [<4h] [4-12h] [12-24h] [1-3j] [3j+]
- Pour chaque tranche : csat_moy, nb_avis

Crée 2 graphiques : distribution des notes, courbe CSAT vs délai.

In [ ]:
# 6a — KPIs satisfaction
%%sql df_sat <<
    SELECT
        -- Votre code ici
    FROM tickets


display(df_sat)

In [ ]:
# 6b — Impact délai sur CSAT
%%sql df_delay_csat <<
    SELECT
        -- Votre code ici
    FROM tickets

display(df_delay_csat)

In [ ]:
# Graphiques : distribution notes + courbe CSAT vs délai
# Votre code ici

### 🧠 Tes observations

- À partir de quel délai de résolution la satisfaction chute-t-elle significativement ?
- Quel est le NPS simplifié ?

*(rédige ici)*

---

# 📌 Conclusion du Notebook 3

👉 Résumer les **3 insights principaux** à présenter à M. Diallo :

1. *(insight 1)*
2. *(insight 2)*
3. *(insight 3)*

---

➡️ **Prochaine étape : Notebook 4 — Dashboard Power BI & Storytelling**

---

*DataProjectLab — apprendre la data sur des cas concrets, structurés et orientés métier.*